## Objectives

The objective of this notebook is to transform the preprocessed data into a machine learning-ready feature matrix.

The feature engineering process includes:

- TF-IDF feature extraction from cleaned job descriptions
- Encoding categorical metadata
- Combining text and structured features
- Feature selection
- Train-test split
- Saving the engineered datasets for model training

In [2]:
# ==========================================================
# Import Libraries
# ==========================================================

import pandas as pd
import numpy as np

from scipy.sparse import hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

import joblib

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# ==========================================================
# Load Processed Dataset
# ==========================================================

df = pd.read_csv("../data/processed/processed_jobs.csv")

print("Processed dataset loaded successfully!")

print(f"\nRows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

df.head()

Processed dataset loaded successfully!

Rows    : 17879
Columns : 18


,clean_text,text_length,word_count,avg_word_length,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,salary_range_missing,department_missing,company_profile_missing,requirements_missing,benefits_missing,fraudulent
0,marketing intern food weve created groundbreak...,2003,242,7.280992,0,1,0,Other,Internship,Unknown,Unknown,Marketing,1,0,0,0,1,0
1,customer service cloud video production second...,4088,529,6.729679,0,1,0,Full-time,Not Applicable,Unknown,Marketing and Advertising,Customer Service,1,0,0,0,0,0
2,commissioning machinery assistant cma valor se...,1994,218,8.151376,0,1,0,Unknown,Unknown,Unknown,Unknown,Unknown,1,1,0,0,1,0
3,account executive washington dc passion improv...,4433,479,8.256785,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,1,0,0,0,0,0
4,bill review manager spotsource solution llc gl...,3220,354,8.098870,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,1,1,0,0,0,0


In [4]:
# ==========================================================
# Dataset Information
# ==========================================================

print("=" * 60)
print("Processed Dataset Information")
print("=" * 60)

df.info()

print("\nMissing Values\n")

print(df.isnull().sum())

Processed Dataset Information
<class 'pandas.DataFrame'>
RangeIndex: 17879 entries, 0 to 17878
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   clean_text               17879 non-null  str    
 1   text_length              17879 non-null  int64  
 2   word_count               17879 non-null  int64  
 3   avg_word_length          17879 non-null  float64
 4   telecommuting            17879 non-null  int64  
 5   has_company_logo         17879 non-null  int64  
 6   has_questions            17879 non-null  int64  
 7   employment_type          17879 non-null  str    
 8   required_experience      17879 non-null  str    
 9   required_education       17879 non-null  str    
 10  industry                 17879 non-null  str    
 11  function                 17879 non-null  str    
 12  salary_range_missing     17879 non-null  int64  
 13  department_missing       17879 non-null  int64  
 14  com

In [5]:
# ==========================================================
# Separate Features and Target
# ==========================================================

X = df.drop(columns=["fraudulent"])

y = df["fraudulent"]

print("Features Shape :", X.shape)
print("Target Shape   :", y.shape)

Features Shape : (17879, 17)
Target Shape   : (17879,)


In [6]:
# ==========================================================
# TF-IDF Feature Extraction
# ==========================================================

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95
)

X_text = tfidf.fit_transform(df["clean_text"])

print("TF-IDF Feature Matrix Shape:")
print(X_text.shape)

TF-IDF Feature Matrix Shape:
(17879, 5000)


In [7]:
# ==========================================================
# Display Sample TF-IDF Features
# ==========================================================

feature_names = tfidf.get_feature_names_out()

print("Total TF-IDF Features:", len(feature_names))

print("\nFirst 30 Features:")

print(feature_names[:30])

Total TF-IDF Features: 5000

First 30 Features:
['aa' 'aabbf' 'aabbf ca' 'ab' 'ab da' 'ab ea' 'abc' 'abc supply' 'ability'
 'ability adapt' 'ability build' 'ability communicate'
 'ability effectively' 'ability learn' 'ability manage'
 'ability multitask' 'ability prioritize' 'ability take' 'ability work'
 'able' 'able multitask' 'able perform' 'able work' 'abroad'
 'abroad conversational' 'abroad help' 'abroad play' 'absolutely' 'ac'
 'ac ca']


In [8]:
# ==========================================================
# One-Hot Encode Categorical Features
# ==========================================================

categorical_columns = [
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_categorical = encoder.fit_transform(df[categorical_columns])

print("=" * 60)
print("Categorical Feature Encoding")
print("=" * 60)

print("Categorical Feature Matrix Shape:", X_categorical.shape)
print("Total Encoded Features:", len(encoder.get_feature_names_out()))

Categorical Feature Encoding
Categorical Feature Matrix Shape: (17879, 198)
Total Encoded Features: 198


In [9]:
# ==========================================================
# Select Numerical Features
# ==========================================================

numerical_columns = [
    "text_length",
    "word_count",
    "avg_word_length",
    "telecommuting",
    "has_company_logo",
    "has_questions",
    "salary_range_missing",
    "department_missing",
    "company_profile_missing",
    "requirements_missing",
    "benefits_missing"
]

X_numerical = df[numerical_columns]

print("=" * 60)
print("Numerical Features")
print("=" * 60)

print("Shape:", X_numerical.shape)

X_numerical.head()

Numerical Features
Shape: (17879, 11)


,text_length,word_count,avg_word_length,telecommuting,has_company_logo,has_questions,salary_range_missing,department_missing,company_profile_missing,requirements_missing,benefits_missing
0,2003,242,7.280992,0,1,0,1,0,0,0,1
1,4088,529,6.729679,0,1,0,1,0,0,0,0
2,1994,218,8.151376,0,1,0,1,1,0,0,1
3,4433,479,8.256785,0,1,0,1,0,0,0,0
4,3220,354,8.098870,0,1,1,1,1,0,0,0


In [10]:
# ==========================================================
# Combine TF-IDF + Categorical + Numerical Features
# ==========================================================

X_final = hstack([
    X_text,
    X_categorical,
    X_numerical.values
])

print("=" * 60)
print("Final Feature Matrix")
print("=" * 60)

print("Shape:", X_final.shape)

Final Feature Matrix
Shape: (17879, 5209)


In [13]:
# ==========================================================
# Train-Test Split
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

split_summary = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "Samples": [len(y_train), len(y_test)],
    "Percentage": [
        round(len(y_train) / len(y) * 100, 2),
        round(len(y_test) / len(y) * 100, 2)
    ],
    "Features": [X_train.shape[1], X_test.shape[1]]
})

class_distribution = pd.DataFrame({
    "Training": y_train.value_counts().sort_index(),
    "Testing": y_test.value_counts().sort_index()
})

class_distribution.index = ["Real Job (0)", "Fake Job (1)"]

print("Train-Test Split Summary")
display(split_summary)

print("Class Distribution")
display(class_distribution)

Train-Test Split Summary


,Dataset,Samples,Percentage,Features
0,Training,14303,80.0,5209
1,Testing,3576,20.0,5209


Class Distribution


,Training,Testing
Real Job (0),13611,3403
Fake Job (1),692,173


In [14]:
# ==========================================================
# Save All Feature Engineering Outputs
# ==========================================================

# Save train-test datasets
joblib.dump(X_train, "../data/processed/X_train.pkl")
joblib.dump(X_test, "../data/processed/X_test.pkl")
joblib.dump(y_train, "../data/processed/y_train.pkl")
joblib.dump(y_test, "../data/processed/y_test.pkl")

# Save feature engineering objects
joblib.dump(tfidf, "../data/processed/tfidf_vectorizer.pkl")
joblib.dump(encoder, "../data/processed/onehot_encoder.pkl")

# Summary table
saved_files = pd.DataFrame({
    "File Name": [
        "X_train.pkl",
        "X_test.pkl",
        "y_train.pkl",
        "y_test.pkl",
        "tfidf_vectorizer.pkl",
        "onehot_encoder.pkl"
    ],
    "Purpose": [
        "Training Features",
        "Testing Features",
        "Training Labels",
        "Testing Labels",
        "TF-IDF Vectorizer",
        "One-Hot Encoder"
    ]
})

print("=" * 60)
print("All Feature Engineering Outputs Saved Successfully")
print("=" * 60)

display(saved_files)

All Feature Engineering Outputs Saved Successfully


,File Name,Purpose
0,X_train.pkl,Training Features
1,X_test.pkl,Testing Features
2,y_train.pkl,Training Labels
3,y_test.pkl,Testing Labels
4,tfidf_vectorizer.pkl,TF-IDF Vectorizer
5,onehot_encoder.pkl,One-Hot Encoder
